In [ ]:
# ==========================================
# 1. ตั้งค่า Kaggle API และดาวน์โหลดข้อมูล
# ==========================================
from google.colab import files
import os

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("กรุณาอัปโหลดไฟล์ kaggle.json ของคุณ:")
    uploaded = files.upload()
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("ตั้งค่า Kaggle API สำเร็จ")

# ดาวน์โหลดข้อมูลและแตกไฟล์
!kaggle competitions download -c super-ai-engineer-ss-6-sleep-stage-classification
!unzip -q -o super-ai-engineer-ss-6-sleep-stage-classification.zip -d data_files

# ==========================================
# 2. นำเข้าไลบรารีและฟังก์ชันสกัดคุณลักษณะ
# ==========================================
import pandas as pd
import numpy as np
import glob
from tqdm.auto import tqdm
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

def extract_features(df):
    features = {}
    sensors = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'TEMP', 'EDA', 'HR']
    for col in sensors:
        if col in df.columns:
            features[f'{col}_mean'] = df[col].mean()
            features[f'{col}_std'] = df[col].std()
            features[f'{col}_skew'] = df[col].skew()
            features[f'{col}_range'] = df[col].max() - df[col].min()
    return pd.Series(features)

# ==========================================
# 3. โหลดและเตรียมข้อมูล (แก้บัคเรื่อง ID ซ้ำเรียบร้อย)
# ==========================================
# ครอบคลุมทั้งกรณีที่ไฟล์อยู่ในโฟลเดอร์ชั้นเดียวและสองชั้น
TRAIN_FILES = sorted(glob.glob('data_files/train/train/*.csv') + glob.glob('data_files/train/*.csv'))
TEST_FILES = sorted(glob.glob('data_files/**/test*/**/*.csv') + glob.glob('data_files/test/*.csv'))

label_mapping = {'W': 0, 'N1': 1, 'N2': 2, 'N3': 3, 'R': 4}
inv_label_mapping = {v: k for k, v in label_mapping.items()}

def process_files(file_list, is_train=True):
    features_list = []
    labels = []
    ids = []

    for file in tqdm(file_list):
        df = pd.read_csv(file)
        base_name = os.path.basename(file).replace('.csv', '')

        for i in range(0, len(df), 480):
            chunk = df.iloc[i : i + 480]
            if len(chunk) < 480: continue

            feats = extract_features(chunk)
            features_list.append(feats)

            if is_train:
                # Train set ใช้โหมดเพื่อหาคำตอบและสร้าง ID ชั่วคราว
                ids.append(f"{base_name}_{i//480}")
                label_str = chunk['Sleep_Stage'].mode()[0]
                labels.append(label_mapping[label_str])
            else:
                # *** จุดที่แก้ปัญหา: Test Set จะใช้ base_name เพียวๆ เป็น ID ตาม Sample Submission ***
                ids.append(base_name)

    return pd.DataFrame(features_list), np.array(labels) if is_train else None, ids

print("กำลังประมวลผล Train Data...")
X_train, y_train, _ = process_files(TRAIN_FILES, is_train=True)

# ==========================================
# 4. ฝึกสอนโมเดล
# ==========================================
print("กำลังฝึกสอนโมเดล LightGBM...")
model = lgb.LGBMClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# ==========================================
# 5. ทำนายผลและบันทึกไฟล์
# ==========================================
print("กำลังประมวลผล Test Data และทำนายผล...")
X_test, _, test_ids = process_files(TEST_FILES, is_train=False)
test_preds = model.predict(X_test)
final_labels = [inv_label_mapping[p] for p in test_preds]

# สร้างตารางข้อมูลเตรียมส่ง
submission = pd.DataFrame({'id': test_ids, 'labels': final_labels})

# *** จุดที่แก้ปัญหาที่ 2: ลบแถวที่ ID ซ้ำกันทิ้ง เพื่อให้เหลือ 7,832 แถวถ้วน ***
submission = submission.drop_duplicates(subset=['id']).reset_index(drop=True)

submission.to_csv('submission_lgbm.csv', index=False)
print(f"จำนวนแถวที่ส่งทั้งหมด: {len(submission)} แถว (ต้องเท่ากับ 7832)")
print("บันทึกไฟล์ submission_lgbm.csv เรียบร้อยแล้ว! นำไปส่ง Kaggle ได้เลย")

super-ai-engineer-ss-6-sleep-stage-classification.zip: Skipping, found more recently modified local copy (use --force to force download)
กำลังประมวลผล Train Data...


  0%|          | 0/83 [00:00<?, ?it/s]

กำลังฝึกสอนโมเดล LightGBM...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005708 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7140
[LightGBM] [Info] Number of data points in the train set: 66745, number of used features: 28
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
กำลังประมวลผล Test Data และทำนายผล...


  0%|          | 0/7832 [00:00<?, ?it/s]

จำนวนแถวที่ส่งทั้งหมด: 7832 แถว (ต้องเท่ากับ 7832)
บันทึกไฟล์ submission_lgbm.csv เรียบร้อยแล้ว! นำไปส่ง Kaggle ได้เลย
